# Exercise 5

In [31]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

## part 1

In [39]:
def confidence_interval(data, alpha = 0.05):
    n = len(data)
    mean = np.mean(data)
    std_err = np.std(data, ddof=1) / np.sqrt(n)
    t_score = stats.t.ppf(1 - alpha/2, df=n-1)
    margin_of_error = t_score * std_err
    return mean, mean - margin_of_error, mean + margin_of_error

In [40]:
#monte carlo estimator
np.random.seed(42)
samples = 100
U = np.random.uniform(0, 1, samples)
X = np.exp(U)

mean_MC, lower_MC, upper_MC = confidence_interval(X)
print(f"Mean (point estimate): {mean_MC}, 95% CI: [{lower_MC}, {upper_MC}]")

Mean (point estimate): 1.6720612537554935, 95% CI: [1.5728239057161093, 1.7712986017948777]


## part 2

In [41]:
f = lambda x: np.exp(x)

Y_av = (f(U) + f(1-U))/2

mean_AV, lower_AV, upper_AV = confidence_interval(Y_av)
print(f"Mean (point estimate): {mean_AV}, 95% CI: [{lower_AV}, {upper_AV}]")

Mean (point estimate): 1.7226021253193073, 95% CI: [1.7101879016931159, 1.7350163489454988]


## part 3

In [42]:
#choose Z to me uniform distribution 
np.random.seed(42)
Z = U
mu_z = 0.5

c = - np.cov(X,Z)[0,1]/ np.var(Z)

Y_cv =  X + c*(Z - mu_z)

mean_CV, lower_CV, upper_CV = confidence_interval(Y_cv)
print(f"Mean (point estimate): {mean_CV}, 95% CI: [{lower_CV}, {upper_CV}]")

Mean (point estimate): 1.7223070150700954, 95% CI: [1.7099399890438187, 1.7346740410963721]


## part 4

In [43]:
Y_strat = []

for i in range(1, samples + 1):
    U_strat = np.random.uniform((i-1)/samples, (i)/samples, samples//samples)

    Y_strat.append(np.mean(np.exp(U_strat)))

mean_strat, lower_strat, upper_strat = confidence_interval(Y_strat)

print(f"Mean (point estimate): {mean_strat}, 95% CI: [{lower_strat}, {upper_strat}]")


Mean (point estimate): 1.7177942727413946, 95% CI: [1.6197064317739234, 1.8158821137088659]


## part 5

In [60]:
lamb = 1
m_servers = 10
mean_service_time = 8
mean_time_between_customers = 1
customers = 10000
n_reps = 10

In [66]:
np.random.seed(42)
def simulate_queue(arrivals_distribution, service_distribution, customers, m_servers, n_reps):
    blocking_fractions = []
    
    inbetween_arrivals = arrivals_distribution(customers)
    service_times = service_distribution(customers)
    arrival_times = np.cumsum(inbetween_arrivals)

    service_free_times = np.zeros(m_servers)
    blocked = 0

    for i in range(customers):
        arr = arrival_times[i]

        free_servers = service_free_times <= arr

        if np.any(free_servers):
            server_index = np.argmax(free_servers)
            service_free_times[server_index] = arr + service_times[i]
        else:
            blocked += 1
    
    blocking_fraction = blocked / customers

    blocking_fractions.append(blocking_fraction)
    return blocking_fractions, np.mean(inbetween_arrivals)


Y_block = []
Z_arr = []
mu_z = 1

def arr_poisson(customers):
    return np.random.exponential(mean_time_between_customers, size=customers)

def service_exp(customers):
    return np.random.exponential(mean_service_time, size=customers)


for _ in range(n_reps):
    b, s = simulate_queue(arr_poisson, service_exp, customers, m_servers, n_reps)
    Y_block.append(b)
    Z_arr.append(s)

Y_block = np.array(Y_block)
Z_arr = np.array(Z_arr)

# Apply Control Variate using service time
Y_block = np.asarray(Y_block).ravel()
Z_arr = np.asarray(Z_arr).ravel()
cov_mat = np.cov(Y_block, Z_arr)
c_star_queue = -cov_mat[0, 1] / cov_mat[1, 1]
y_cv_queue = Y_block + c_star_queue * (Z_arr - mu_z)

mean_cv, ci_low_cv, ci_high_cv = confidence_interval(y_cv_queue)

print(f"CV Blocking Estimate: {mean_cv:.4f} (95% CI: [{ci_low_cv:.4f}, {ci_high_cv:.4f}])")



CV Blocking Estimate: 0.1230 (95% CI: [0.1191, 0.1270])


## part 6

In [84]:
np.random.seed(42)

def simulate_queue(arrivals, service_times, customers, m_servers):
    arrival_times = np.cumsum(arrivals)
    service_free_times = np.zeros(m_servers)
    blocked = 0

    for i in range(customers):
        arr = arrival_times[i]

        free_servers = service_free_times <= arr

        if np.any(free_servers):
            server_index = np.argmax(free_servers)
            service_free_times[server_index] = arr + service_times[i]
        else:
            blocked += 1

    return blocked / customers

p1 = 0.8
lambda1 = 0.8333
p2 = 0.2
lambda2 = 5

Y_block_pos = []
Y_block_hyper = []
Y_block_pos_indep = []
for _ in range(n_reps):
    # common random numbers within each replication
    u_arr_crn = np.random.rand(customers)
    u_srv_crn = np.random.rand(customers)
    u_coin_crn = np.random.rand(customers)

    arrivals_poisson = -(1 / 1) * np.log(u_arr_crn)

    arrivals_hyper = np.where(
        u_coin_crn <= p1,
        -(1 / lambda1) * np.log(u_arr_crn),
        -(1 / lambda2) * np.log(u_arr_crn)
    )

    service_exp_times = -mean_service_time * np.log(u_srv_crn)

    b_pos = simulate_queue(arrivals_poisson, service_exp_times, customers, m_servers)
    b_hyper = simulate_queue(arrivals_hyper, service_exp_times, customers, m_servers)

    Y_block_pos.append(b_pos)
    Y_block_hyper.append(b_hyper)

    u_arr_indep = np.random.rand(customers)
    u_srv_indep = np.random.rand(customers)
    block_pois_indep = simulate_queue(u_arr_indep, u_srv_indep, customers, m_servers)
    Y_block_pos_indep.append(block_pois_indep)



diff_notsame = np.array(Y_block_pos_indep) - np.array(Y_block_hyper)
diff_same = np.array(Y_block_pos) - np.array(Y_block_hyper)

print(f"Mean difference (same CRN): {diff_same.mean():.6f}, Variance: {diff_same.var(ddof=1):.6f}")
print(f"Mean difference (independent): {diff_notsame.mean():.6f}, Variance: {diff_notsame.var(ddof=1):.6f}")

Mean difference (same CRN): -0.016370, Variance: 0.000024
Mean difference (independent): -0.135430, Variance: 0.000088


## part 7

In [89]:
np.random.seed(42)
def crude_mc(a, n):
    z = np.random.normal(0, 1, n)
    indicators = z > a
    estimate = np.mean(indicators)
    var = np.var(indicators, ddof=1)
    return estimate, var


def importance_sampling(a, n, sigma2=1):
    sigma = np.sqrt(sigma2)

    x = np.random.normal(a, sigma, n)

    f = stats.norm.pdf(x, loc=0, scale=1)
    g = stats.norm.pdf(x, loc=a, scale=sigma)

    weights = f / g
    values = (x > a) * weights

    estimate = np.mean(values)
    var = np.var(values, ddof=1)

    return estimate, var

In [91]:
sigma =1
a = [2,4]
n = 10000


for _ in a:
    importance_estimate, importance_var = importance_sampling(_, n, sigma2=sigma**2)
    crude_estimate, crude_var = crude_mc(_, n)
    print(f"a={_}:")
    print(f"  Importance Sampling -> Estimate: {importance_estimate:.6e}, Var: {importance_var:.6e}")
    print(f"  Crude MC            -> Estimate: {crude_estimate:.6e}, Var: {crude_var:.6e}")
    print()


a=2:
  Importance Sampling -> Estimate: 2.290045e-02, Var: 1.220263e-03
  Crude MC            -> Estimate: 2.410000e-02, Var: 2.352154e-02

a=4:
  Importance Sampling -> Estimate: 3.273009e-05, Var: 4.671415e-09
  Crude MC            -> Estimate: 0.000000e+00, Var: 0.000000e+00



In [92]:
np.random.seed(42)

a_values = [2, 4]
n_values = [1000, 10000, 100000]

for a in a_values:
    true_value = 1 - stats.norm.cdf(a)

    print(f"\na = {a}")
    print(f"True value: {true_value:.8f}")

    for n in n_values:
        crude_est, crude_se = crude_mc(a, n)
        is_est, is_se = importance_sampling(a, n, sigma2=1)

        print(f"n = {n}")
        print(f"  Crude MC: {crude_est:.8f}, SE = {crude_se:.8f}")
        print(f"  IS:       {is_est:.8f}, SE = {is_se:.8f}")


a = 2
True value: 0.02275013
n = 1000
  Crude MC: 0.02400000, SE = 0.02344745
  IS:       0.02299193, SE = 0.00119131
n = 10000
  Crude MC: 0.02210000, SE = 0.02161375
  IS:       0.02320876, SE = 0.00125264
n = 100000
  Crude MC: 0.02299000, SE = 0.02246168
  IS:       0.02278856, SE = 0.00121539

a = 4
True value: 0.00003167
n = 1000
  Crude MC: 0.00000000, SE = 0.00000000
  IS:       0.00002768, SE = 0.00000000
n = 10000
  Crude MC: 0.00000000, SE = 0.00000000
  IS:       0.00003163, SE = 0.00000000
n = 100000
  Crude MC: 0.00001000, SE = 0.00001000
  IS:       0.00003179, SE = 0.00000000
